# Cánticos del fútbol colombiano · PLN en cuatro fases

**Pregunta.** ¿Cómo construir un corpus auditable de cánticos y usar representaciones y modelos de lenguaje para explorar similitud, afecto, dimensiones discursivas y entidades?

Trabajo académico en equipo con participación de Manuela Zapata Jiménez, **Procesamiento y Entendimiento del Lenguaje Natural**, Universidad Nacional de Colombia. [Resumen de construcción del corpus](../analisis.ipynb)

Las celdas ejecutables verifican los agregados de la primera fase y prueban métodos de diversidad, similitud y puntuación sobre ejemplos inventados. A continuación se conserva código de las cuatro fases y se explican las decisiones y resultados del experimento. La ejecución completa requiere el corpus académico y sus matrices de revisión, además de NLTK, pandas, scikit-learn, PyTorch, Sentence Transformers y Transformers con los modelos fijados en la entrega.

## 1. Adquisición y controles del corpus

La primera fase registró procedencia, cobertura por club y duplicados exactos. Una representación para comparar textos evita alterar la versión que se conserva como evidencia. Los datos del repositorio contienen conteos agregados de esta fase.

In [1]:
from csv import DictReader
from pathlib import Path

raiz = Path.cwd()
if raiz.name == 'tecnico':
    raiz = raiz.parent

def leer_tabla(nombre):
    with (raiz / 'datos' / nombre).open(encoding='utf-8', newline='') as archivo:
        return list(DictReader(archivo))

calidad = {fila['indicador']: int(fila['conteo']) for fila in leer_tabla('calidad_del_corpus.csv')}
cobertura = leer_tabla('cobertura_por_club.csv')
total = sum(int(fila['canticos_integrados']) for fila in cobertura)
assert calidad['capturas_crudas'] - calidad['duplicados_exactos_eliminados'] == total == 600
assert sum(int(fila['canticos_integrados']) > 0 for fila in cobertura) == 17
print(f'{total} textos integrados · {len(cobertura)} clubes de referencia · 17 con textos')

600 textos integrados · 20 clubes de referencia · 17 con textos


**Carga de tablas y comprobación de columnas necesarias**

```python
def cargar_csv(nombre: str, columnas_obligatorias: set[str] | None = None) -> pd.DataFrame:
    ruta = DATA / nombre
    if not ruta.exists():
        raise FileNotFoundError(f"Falta el archivo requerido: {ruta}")
    tabla = pd.read_csv(ruta, encoding="utf-8-sig", low_memory=False)
    faltantes = (columnas_obligatorias or set()) - set(tabla.columns)
    if faltantes:
        raise ValueError(f"{nombre} no contiene las columnas: {sorted(faltantes)}")
    return tabla

tablas = {
    "canticos": cargar_csv("canticos.csv", {"cantico_id", "club_id", "letra_original", "fuente", "url", "fecha_consulta"}),
    "clubes": cargar_csv("clubes.csv", {"club_id", "nombre_corto", "region"}),
    "cobertura": cargar_csv("control_cobertura.csv", {"club_id", "registros_finales", "estado_primera_pasada"}),
    "duplicados": cargar_csv("duplicados_exactos.csv"),
    "fuentes": cargar_csv("fuentes_revisadas.csv"),
    "cambios": cargar_csv("cambios_identidad_clubes.csv"),
    "bibliografia": cargar_csv("bibliografia_apa7.csv"),
    "decisiones": cargar_csv(
        "decisiones_revision_fase2.csv",
        {"revision_id", "tipo_revision", "canticos_involucrados",
         "decision_revision", "justificacion", "estado_aplicacion"},
    ),
}
canticos = tablas["canticos"]
clubes = tablas["clubes"]
cobertura = tablas["cobertura"]
duplicados = tablas["duplicados"]
fuentes = tablas["fuentes"]
cambios = tablas["cambios"]
bibliografia = tablas["bibliografia"]
decisiones = tablas["decisiones"]

canticos["fecha_consulta"] = pd.to_datetime(canticos["fecha_consulta"], errors="raise")
canticos["numero_palabras"] = pd.to_numeric(canticos["numero_palabras"], errors="raise").astype("int64")
print("Tablas cargadas:", {nombre: tabla.shape for nombre, tabla in tablas.items()})
```

**Control de integridad del corpus adquirido**

```python
# Controles cuantitativos que respaldan la afirmación anterior
campos_nucleo = ["cantico_id", "club_id", "titulo_referencia", "letra_original", "fuente", "url", "fecha_consulta", "metodo_adquisicion"]
control_calidad = pd.DataFrame({
    "Control": [
        "Campos núcleo completos", "IDs únicos", "URLs únicas", "Letras no vacías",
        "Contextos conservados", "Referentes verificados", "Bandera de calidad", "Identidad histórica"
    ],
    "Resultado": [
        not canticos[campos_nucleo].isna().any().any(), canticos["cantico_id"].is_unique,
        canticos["url"].is_unique, canticos["letra_original"].str.strip().ne("").all(),
        canticos["contexto_cantico"].notna().sum(),
        canticos["estado_contexto"].eq("verificado con fuente externa").sum(),
        canticos["revision_manual"].eq("Sí").sum(),
        canticos["equipo_mencionado_fuente"].isin(["Alianza Petrolera", "La Equidad"]).sum(),
    ]
})
display(control_calidad)
```

## 2. Limpieza, tokenización y exploración

El proyecto separa la versión de procedencia, la preparación destinada a modelos y la representación léxica de análisis. La comparación entre TTR y MATTR controla cuánto cambia la diversidad observada por la longitud de cada cántico.

In [2]:
import numpy as np

def mattr(tokens: list[str], ventana: int = 25) -> float:
    tokens = [token.casefold() for token in tokens]
    if not tokens:
        return np.nan
    ancho = min(ventana, len(tokens))
    valores = [
        len(set(tokens[i : i + ancho])) / ancho
        for i in range(len(tokens) - ancho + 1)
    ]
    return float(np.mean(valores))

In [3]:
palabras = ['equipo', 'barrio', 'equipo', 'fiesta']
diversidad = mattr(palabras, ventana=3)
assert np.isclose(diversidad, 5 / 6)
assert np.isnan(mattr([]))
print(f'MATTR de ejemplo inventado (ventana 3): {diversidad:.3f}')

MATTR de ejemplo inventado (ventana 3): 0.833


**Tokenización auditable y filtros léxicos**

```python
def tokenizar_auditable(texto: str) -> list[str]:
    """Tokenización superficial de NLTK compatible con el conteo de adquisición."""
    return TOKENIZADOR_NLTK.tokenize(str(texto))


def tokens_lexicos(texto: str) -> list[str]:
    """Tokens de contenido: NLTK español, sin stopwords y con negaciones preservadas."""
    tokens = [token.casefold() for token in tokenizar_auditable(texto)]
    return [
        token for token in tokens
        if len(token) > 1
        and any(caracter.isalpha() for caracter in token)
        and token not in STOPWORDS_ANALISIS
    ]
```

**Normalización diferenciada para el modelo y el análisis léxico**

```python
def eliminar_marcadores_editoriales(texto: str) -> str:
    return PATRON_MARCADOR_EDITORIAL.sub(" ", str(texto))


def reemplazar_tokens_asterisco(texto: str) -> str:
    def resolver(coincidencia):
        original = coincidencia.group(0)
        reemplazo = REEMPLAZOS_ASTERISCO.get(original.casefold())
        if reemplazo is None:
            return original
        if original.isupper():
            return reemplazo.upper()
        if original[:1].isupper():
            return reemplazo.capitalize()
        return reemplazo

    return PATRON_TOKEN_ASTERISCO.sub(resolver, str(texto))


def normalizar_para_modelo(texto: str) -> str:
    # Limpieza conservadora: preserva el contenido y retira metadatos editoriales.
    texto = html.unescape(str(texto))
    texto = unicodedata.normalize("NFC", texto)
    texto = texto.translate(CARACTERES_INVISIBLES)
    texto = texto.replace("\r\n", "\n").replace("\r", "\n")
    texto = eliminar_marcadores_editoriales(texto)
    texto = reemplazar_tokens_asterisco(texto)
    lineas = [re.sub(r"[ \t]+", " ", linea).strip() for linea in texto.split("\n")]
    texto = "\n".join(lineas)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


def normalizar_para_eda_lexico(texto: str) -> str:
    # Vista derivada para frecuencias; no es la entrada automática al modelo.
    return " ".join(tokens_lexicos(normalizar_para_modelo(texto)))
```

## 3. Representaciones y revisión de similitud

Se comparan TF-IDF por palabras, TF-IDF por caracteres y representaciones contextuales. La similitud coseno propone pares para revisión. Las decisiones revisadas se representan como relaciones de equivalencia cuando corresponde.

In [4]:
class ComponentesEquivalencia:
    def __init__(self):
        self.padre = {}

    def encontrar(self, elemento):
        self.padre.setdefault(elemento, elemento)
        if self.padre[elemento] != elemento:
            self.padre[elemento] = self.encontrar(self.padre[elemento])
        return self.padre[elemento]

    def unir(self, elemento_1, elemento_2):
        raiz_1 = self.encontrar(elemento_1)
        raiz_2 = self.encontrar(elemento_2)
        if raiz_1 != raiz_2:
            self.padre[raiz_2] = raiz_1

In [5]:
grupos = ComponentesEquivalencia()
grupos.unir('ejemplo-a', 'ejemplo-b')
grupos.unir('ejemplo-b', 'ejemplo-c')
assert grupos.encontrar('ejemplo-a') == grupos.encontrar('ejemplo-c')
assert grupos.encontrar('ejemplo-d') != grupos.encontrar('ejemplo-a')
print('Ejemplo inventado · tres elementos vinculados y uno independiente')

Ejemplo inventado · tres elementos vinculados y uno independiente


**Vectorización léxica, inspección de matrices y vocabulario**

```python
textos_lexicos_f3 = corpus_preparado["texto_lexico_filtrado"].fillna("")
textos_modelo_f3 = corpus_preparado["texto_modelo"].fillna("")

vectorizador_palabras = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95,
    sublinear_tf=True,
    smooth_idf=True,
    norm="l2",
    dtype=np.float32,
)
X_tfidf_palabras = vectorizador_palabras.fit_transform(textos_lexicos_f3)

vectorizador_caracteres = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.98,
    max_features=8000,
    sublinear_tf=True,
    smooth_idf=True,
    norm="l2",
    dtype=np.float32,
)
X_tfidf_caracteres = vectorizador_caracteres.fit_transform(textos_modelo_f3)

def tokens_minuscula(texto):
    return [token.casefold() for token in tokenizar_auditable(texto)]

descriptores_estructurales = pd.DataFrame(
    {
        "n_tokens_modelo": corpus_preparado["n_tokens_modelo"],
        "n_lineas_modelo": textos_modelo_f3.str.count("\n") + 1,
        "tokens_por_linea_modelo": corpus_preparado["n_tokens_modelo"]
        / (textos_modelo_f3.str.count("\n") + 1),
        "mattr_25_modelo": textos_modelo_f3.map(
            lambda texto: mattr(tokenizar_auditable(texto))
        ),
        "proporcion_lineas_repetidas_modelo": textos_modelo_f3.map(
            proporcion_lineas_repetidas
        ),
        "proporcion_tokens_lexicos_modelo": (
            corpus_preparado["n_tokens_lexicos_filtrados"]
            / corpus_preparado["n_tokens_modelo"]
        ),
        "proporcion_negaciones_modelo": textos_modelo_f3.map(
            lambda texto: sum(
                token in NEGACIONES_PRESERVADAS for token in tokens_minuscula(texto)
            ) / max(len(tokenizar_auditable(texto)), 1)
        ),
        "exclamaciones_por_100_tokens_modelo": [
            (texto.count("!") + texto.count("¡")) / max(n_tokens, 1) * 100
            for texto, n_tokens in zip(
                textos_modelo_f3, corpus_preparado["n_tokens_modelo"]
            )
        ],
        "proporcion_mayusculas_modelo": textos_modelo_f3.map(
            proporcion_mayusculas
        ),
        "alargamientos_por_1000_tokens_modelo": [
            sum(
                bool(re.search(r"([a-záéíóúüñ])\1{2,}", token.casefold()))
                for token in tokenizar_auditable(texto)
            ) / max(n_tokens, 1) * 1000
            for texto, n_tokens in zip(
                textos_modelo_f3, corpus_preparado["n_tokens_modelo"]
            )
        ],
    }
)
escalador_estructural = StandardScaler()
X_estructural = escalador_estructural.fit_transform(
    descriptores_estructurales
).astype(np.float32)

def memoria_matriz_dispersa_mb(matriz):
    return (matriz.data.nbytes + matriz.indices.nbytes + matriz.indptr.nbytes) / 2**20

def perfil_matriz_dispersa(nombre, matriz):
    normas = np.sqrt(matriz.multiply(matriz).sum(axis=1)).A1
    return {
        "representacion": nombre,
        "filas": matriz.shape[0],
        "caracteristicas": matriz.shape[1],
        "valores_no_cero": matriz.nnz,
        "densidad_pct": matriz.nnz / (matriz.shape[0] * matriz.shape[1]) * 100,
        "filas_vacias": int((matriz.getnnz(axis=1) == 0).sum()),
        "norma_l2_min": normas.min(),
        "norma_l2_max": normas.max(),
        "memoria_mb": memoria_matriz_dispersa_mb(matriz),
    }

perfil_sparse = pd.DataFrame(
    [
        perfil_matriz_dispersa("TF-IDF palabras 1–2", X_tfidf_palabras),
        perfil_matriz_dispersa("TF-IDF caracteres 3–5", X_tfidf_caracteres),
    ]
)

def inventario_vocabulario(vectorizador, matriz):
    return pd.DataFrame(
        {
            "caracteristica": vectorizador.get_feature_names_out(),
            "frecuencia_documental": np.asarray((matriz > 0).sum(axis=0)).ravel(),
            "tfidf_promedio": np.asarray(matriz.mean(axis=0)).ravel(),
        }
    ).sort_values(["tfidf_promedio", "frecuencia_documental"], ascending=False)

vocabulario_palabras = inventario_vocabulario(
    vectorizador_palabras, X_tfidf_palabras
)
vocabulario_caracteres = inventario_vocabulario(
    vectorizador_caracteres, X_tfidf_caracteres
)
vocabulario_caracteres.insert(
    1,
    "caracteristica_visible",
    vocabulario_caracteres["caracteristica"].str.replace(" ", "␠", regex=False),
)

display(perfil_sparse.round(4))
display(vocabulario_palabras.head(20))
display(
    vocabulario_caracteres[
        ["caracteristica_visible", "frecuencia_documental", "tfidf_promedio"]
    ].head(20)
)
display(
    pd.DataFrame(
        {
            "media_z": X_estructural.mean(axis=0),
            "desviacion_z": X_estructural.std(axis=0),
        },
        index=descriptores_estructurales.columns,
    ).round(4)
)
```

**Representación contextual con modelo y revisión fijados**

```python
MODELO_CONTEXTUAL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODELO_CONTEXTUAL_REVISION = "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
CACHE_MODELO_FASE3 = ROOT / "modelos_huggingface"

inicio_carga_modelo = time.perf_counter()
modelo_contextual = SentenceTransformer(
    MODELO_CONTEXTUAL_ID,
    revision=MODELO_CONTEXTUAL_REVISION,
    device=DISPOSITIVO_FASE3,
    cache_folder=str(CACHE_MODELO_FASE3),
    trust_remote_code=False,
)
tokenizador_contextual = modelo_contextual.tokenizer
limite_modelo = int(modelo_contextual.max_seq_length)
tokens_especiales = int(
    tokenizador_contextual.num_special_tokens_to_add(pair=False)
)
limite_contenido = limite_modelo - tokens_especiales
assert limite_modelo == 128
assert limite_contenido > 0
assert modelo_contextual.get_sentence_embedding_dimension() == 384

textos_fragmentos = []
indices_fragmentos = []
filas_tokenizacion = []
for indice_documento, texto in enumerate(textos_modelo_f3):
    ids = tokenizador_contextual(
        texto,
        add_special_tokens=False,
        truncation=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False,
    )["input_ids"]
    assert ids
    n_unk = (
        ids.count(tokenizador_contextual.unk_token_id)
        if tokenizador_contextual.unk_token_id is not None else 0
    )
    fragmentos_documento = []
    for inicio in range(0, len(ids), limite_contenido):
        ids_fragmento = ids[inicio : inicio + limite_contenido]
        texto_fragmento = tokenizador_contextual.decode(
            ids_fragmento,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        textos_fragmentos.append(texto_fragmento)
        indices_fragmentos.append((indice_documento, len(ids_fragmento)))
        fragmentos_documento.append(len(ids_fragmento))
    filas_tokenizacion.append(
        {
            "cantico_id": corpus_preparado.iloc[indice_documento]["cantico_id"],
            "n_subpalabras": len(ids),
            "n_fragmentos": len(fragmentos_documento),
            "max_subpalabras_fragmento": max(fragmentos_documento),
            "n_tokens_desconocidos": n_unk,
        }
    )

diagnostico_tokenizacion = pd.DataFrame(filas_tokenizacion)
inicio_inferencia = time.perf_counter()
embeddings_fragmentos = modelo_contextual.encode(
    textos_fragmentos,
    batch_size=TAMANO_LOTE_FASE3,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True,
).astype(np.float32)
duracion_inferencia = time.perf_counter() - inicio_inferencia

X_contextual = np.zeros(
    (len(corpus_preparado), embeddings_fragmentos.shape[1]), dtype=np.float32
)
pesos_documento = np.zeros(len(corpus_preparado), dtype=np.float32)
for vector, (indice_documento, peso) in zip(
    embeddings_fragmentos, indices_fragmentos
):
    X_contextual[indice_documento] += vector * peso
    pesos_documento[indice_documento] += peso
X_contextual /= pesos_documento[:, None]
X_contextual /= np.linalg.norm(X_contextual, axis=1, keepdims=True)

resumen_contextual = pd.DataFrame(
    {
        "indicador": [
            "Documentos", "Dimensiones", "Subpalabras totales",
            "Subpalabras medianas", "Subpalabras percentil 95",
            "Subpalabras máximas", "Fragmentos procesados",
            "Textos con más de un fragmento", "Máximo de fragmentos por texto",
            "Tokens desconocidos", "Dispositivo", "Tiempo de inferencia (s)",
        ],
        "valor": [
            X_contextual.shape[0], X_contextual.shape[1],
            diagnostico_tokenizacion["n_subpalabras"].sum(),
            diagnostico_tokenizacion["n_subpalabras"].median(),
            diagnostico_tokenizacion["n_subpalabras"].quantile(0.95),
            diagnostico_tokenizacion["n_subpalabras"].max(),
            len(textos_fragmentos),
            diagnostico_tokenizacion["n_fragmentos"].gt(1).sum(),
            diagnostico_tokenizacion["n_fragmentos"].max(),
            diagnostico_tokenizacion["n_tokens_desconocidos"].sum(),
            NOMBRE_DISPOSITIVO_FASE3, round(duracion_inferencia, 2),
        ],
    }
)
display(resumen_contextual)
display(
    diagnostico_tokenizacion.nlargest(8, "n_subpalabras").merge(
        corpus_preparado[["cantico_id", "nombre_corto", "titulo_referencia"]],
        on="cantico_id", how="left", validate="one_to_one",
    )
)
print(f"Carga del modelo: {time.perf_counter() - inicio_carga_modelo:.2f} s")
```

**Comparación de espacios y pares candidatos**

```python
similitudes = {
    "TF-IDF palabras": cosine_similarity(X_tfidf_palabras),
    "TF-IDF caracteres": cosine_similarity(X_tfidf_caracteres),
    "Embedding contextual": cosine_similarity(X_contextual),
}
triangulo_superior = np.triu_indices(len(corpus_preparado), k=1)

filas_distribucion = []
filas_vecino = []
clubes_array = corpus_preparado["club_id"].to_numpy()
for nombre, matriz_similitud in similitudes.items():
    valores = matriz_similitud[triangulo_superior]
    sin_diagonal = matriz_similitud.copy()
    np.fill_diagonal(sin_diagonal, -np.inf)
    vecino = sin_diagonal.argmax(axis=1)
    filas_distribucion.append(
        {
            "representacion": nombre,
            "mediana": np.median(valores),
            "q75": np.quantile(valores, 0.75),
            "p95": np.quantile(valores, 0.95),
            "p99": np.quantile(valores, 0.99),
            "maximo": valores.max(),
        }
    )
    filas_vecino.append(
        {
            "representacion": nombre,
            "vecino_del_mismo_club_pct": (
                clubes_array[vecino] == clubes_array
            ).mean() * 100,
        }
    )
distribucion_similitud = pd.DataFrame(filas_distribucion)
diagnostico_vecinos = pd.DataFrame(filas_vecino)

def pares_superiores(nombre, matriz_similitud, n=5):
    valores = matriz_similitud[triangulo_superior]
    posiciones = np.argsort(valores)[-n:][::-1]
    filas = []
    for posicion in posiciones:
        i = triangulo_superior[0][posicion]
        j = triangulo_superior[1][posicion]
        fila_i = corpus_preparado.iloc[i]
        fila_j = corpus_preparado.iloc[j]
        filas.append(
            {
                "representacion": nombre,
                "similitud": valores[posicion],
                "cantico_1": fila_i["cantico_id"],
                "club_1": fila_i["nombre_corto"],
                "titulo_1": fila_i["titulo_referencia"],
                "cantico_2": fila_j["cantico_id"],
                "club_2": fila_j["nombre_corto"],
                "titulo_2": fila_j["titulo_referencia"],
                "mismo_club": fila_i["club_id"] == fila_j["club_id"],
            }
        )
    return pd.DataFrame(filas)

pares_cercanos = pd.concat(
    [pares_superiores(nombre, matriz) for nombre, matriz in similitudes.items()],
    ignore_index=True,
)
display(distribucion_similitud.round(4))
display(diagnostico_vecinos.round(2))
display(pares_cercanos)

pca_contextual_2d = PCA(n_components=2, random_state=SEMILLA_FASE3)
coordenadas_contextuales = pca_contextual_2d.fit_transform(X_contextual)
pca_contextual_50 = PCA(n_components=50, random_state=SEMILLA_FASE3).fit(
    X_contextual
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for region in regiones:
    mascara = corpus_preparado["region"].eq(region).to_numpy()
    axes[0].scatter(
        coordenadas_contextuales[mascara, 0],
        coordenadas_contextuales[mascara, 1],
        s=25, alpha=0.55, label=region, color=colores_region[region],
        edgecolors="none",
    )
axes[0].set(
    title=(
        "Embedding contextual · PCA 2D\n"
        f"varianza mostrada: {pca_contextual_2d.explained_variance_ratio_.sum()*100:.1f}%"
    ),
    xlabel="Componente principal 1", ylabel="Componente principal 2",
)
axes[0].legend(title="Región", fontsize=8)

for nombre, matriz_similitud in similitudes.items():
    axes[1].hist(
        matriz_similitud[triangulo_superior], bins=45, density=True,
        alpha=0.45, label=nombre,
    )
axes[1].set(
    title="Distribución de similitud entre pares",
    xlabel="Similitud coseno", ylabel="Densidad",
)
axes[1].legend(fontsize=8)
fig.suptitle(
    "Geometría contextual y diferencias entre representaciones",
    fontsize=15, fontweight="bold",
)
fig.tight_layout()
plt.show()
```

## 4. Afecto y clasificación de múltiples etiquetas sin entrenamiento adicional

La entrega compara clasificadores de sentimiento y emociones con clasificación *zero-shot* para **33 hipótesis**. En NLI se contrasta puntuar la implicación frente a la contradicción o frente a todas las alternativas, incluida la neutralidad. Se seleccionó **BGE-M3** con hipótesis explícitas e implicación frente al resto. La AUC contra marcadores léxicos y el acuerdo entre configuraciones son señales indirectas. Sin etiquetas humanas del corpus no se pueden estimar métricas de desempeño supervisado sobre estos textos.

**Fragmentación, agregación y reglas de probabilidad de implicación**

```python
def fragmentar_por_versos(texto, tokenizador, limite):
    """Divide el texto en tramos contiguos de hasta `limite` subpalabras.

    Prefiere cortar en un salto de línea para no partir versos; solo corta dentro
    de una línea cuando un verso excede por sí mismo el límite. Devuelve
    (inicio, fin, n_subpalabras) sobre el texto original.
    """
    desplazamientos = tokenizador(
        texto, add_special_tokens=False, return_offsets_mapping=True,
        truncation=False, verbose=False,
    )["offset_mapping"]
    n = len(desplazamientos)
    assert n > 0, "texto sin subpalabras"
    if n <= limite:
        return [(0, len(texto), n)]
    frontera = [False] + [
        "\n" in texto[desplazamientos[i - 1][1]:desplazamientos[i][0]] for i in range(1, n)
    ]
    tramos, inicio = [], 0
    while inicio < n:
        fin = min(inicio + limite, n)
        if fin < n:
            cortes = [j for j in range(inicio + 1, fin + 1) if frontera[j]]
            if cortes and cortes[-1] - inicio >= limite // 2:
                fin = cortes[-1]
        while True:
            a, b = desplazamientos[inicio][0], desplazamientos[fin - 1][1]
            n_tramo = len(tokenizador(texto[a:b], add_special_tokens=False, verbose=False)["input_ids"])
            if n_tramo <= limite or fin - inicio == 1:
                break
            fin -= 1
        tramos.append((a, b, n_tramo))
        inicio = fin
    return tramos


def agregar_por_texto(valores_fragmento, fragmentos, agregacion):
    """Agrega filas por fragmento a filas por cántico.

    `media_ponderada` usa el número de subpalabras como peso y conserva una
    distribución que suma uno; `maximo` responde a la pregunta de si alguna parte
    del texto activa la etiqueta.
    """
    posiciones = np.array([f[0] for f in fragmentos])
    pesos = np.array([f[3] for f in fragmentos], dtype=np.float64)
    salida = np.zeros((len(TEXTOS_FASE4), valores_fragmento.shape[1]), dtype=np.float64)
    if agregacion == "media_ponderada":
        np.add.at(salida, posiciones, valores_fragmento * pesos[:, None])
        salida /= np.bincount(posiciones, weights=pesos)[:, None]
    elif agregacion == "maximo":
        salida[:] = -np.inf
        np.maximum.at(salida, posiciones, valores_fragmento)
    else:
        raise ValueError(agregacion)
    return salida


def probabilidades_implicacion(logits, modelo):
    """Dos reglas para convertir los logits NLI en la puntuación de una etiqueta.

    `implicacion_vs_contradiccion` reproduce el modo multi_label del pipeline
    zero-shot de transformers: softmax entre implicación y contradicción,
    descartando la clase neutral. `implicacion_vs_resto` usa el softmax sobre
    todas las clases, de modo que la neutralidad también resta. En modelos de
    dos clases (implicación frente a no implicación) ambas reglas coinciden.
    """
    etiquetas = {v.lower(): k for k, v in modelo.config.id2label.items()}
    implicacion = etiquetas["entailment"]
    completa = torch.softmax(logits, dim=-1)[:, implicacion]
    if "contradiction" in etiquetas:
        par = torch.softmax(logits[:, [etiquetas["contradiction"], implicacion]], dim=-1)[:, 1]
    else:
        par = completa
    return {"implicacion_vs_contradiccion": par, "implicacion_vs_resto": completa}
```

**Efecto de retener la neutralidad.** Los valores inventados muestran por qué conviene explicitar la regla. Con la misma implicación y contradicción, una neutralidad alta reduce la puntuación de la etiqueta cuando se incluye en el denominador.

In [6]:
def softmax(valores):
    logits = np.asarray(valores, dtype=float)
    exponentes = np.exp(logits - logits.max())
    return exponentes / exponentes.sum()

contradiccion, neutral, implicacion = 0.1, 2.0, 0.9
contra_contradiccion = softmax([contradiccion, implicacion])[1]
contra_resto = softmax([contradiccion, neutral, implicacion])[2]
assert 0 < contra_resto < contra_contradiccion < 1
print(f'Implicación frente a contradicción: {contra_contradiccion:.3f}')
print(f'Implicación frente al resto: {contra_resto:.3f}')

Implicación frente a contradicción: 0.690
Implicación frente al resto: 0.225


**Tratamiento de las salidas de modelos de emociones**

```python
def activaciones_emocion(clave, umbral_ficha=True):
    tabla = probabilidades_clasificadores[clave]
    if not metadatos_clasificadores[clave]["multietiqueta"]:
        return pd.get_dummies(tabla.idxmax(axis=1)).reindex(columns=tabla.columns, fill_value=False)
    if clave == "emo_goemotions" and umbral_ficha:
        return tabla.ge(pd.Series(UMBRAL_GOEMOTIONS_ES)[tabla.columns])
    return tabla.ge(0.5)
```

**Indicadores de cobertura, acuerdo y contraste con marcadores**

```python
def configuracion_comparable(clave, formulacion, regla):
    """Configuración equivalente de otro modelo: misma formulación y misma regla."""
    return (clave, formulacion, regla if clases_nli[clave] == 3 else "implicacion_vs_resto")


def metricas_configuracion_zs(clave, formulacion, regla):
    tabla = puntuaciones_zero_shot[(clave, formulacion, regla)]
    prevalencia = tabla.ge(UMBRAL_ZS).mean()
    otras = [c for c in MODELOS_NLI_FASE4 if c != clave]
    acuerdo = np.mean(
        [
            spearmanr(tabla[e], puntuaciones_zero_shot[configuracion_comparable(otra, formulacion, regla)][e]).statistic
            for e in ORDEN_ETIQUETAS_ZS for otra in otras
        ]
    )
    aucs = {e: auc_marcador(tabla[e], e) for e in ETIQUETAS_CON_MARCADOR}
    registro = next(
        r for r in REGISTRO_EJECUCION_FASE4 if r["clave"] == clave and r["configuracion"] == f"zs_{formulacion}"
    )
    return {
        "modelo": MODELOS_FASE4[clave]["id"], "clave": clave, "formulacion": formulacion,
        "regla": regla, "auc_lexica_media": np.nanmean(list(aucs.values())),
        "etiquetas_auc_mayor_0_6": int(sum(v > 0.6 for v in aucs.values())),
        "acuerdo_intermodelo_rho": acuerdo,
        "etiquetas_informativas": int(prevalencia.between(0.02, 0.95).sum()),
        "etiquetas_saturadas": int(prevalencia.gt(0.95).sum()),
        "etiquetas_casi_vacias": int(prevalencia.lt(0.02).sum()),
        "etiquetas_activas_mediana": float(tabla.ge(UMBRAL_ZS).sum(axis=1).median()),
        "prevalencia_media_pct": prevalencia.mean() * 100,
        "zona_incierta_0_4_0_6_pct": tabla.stack().between(0.4, 0.6).mean() * 100,
        "rho_desprecio_emocional_vs_rival": spearmanr(tabla["emocional:desprecio"], tabla["rival:desprecio"]).statistic,
        "hipotesis_repetidas": len(ORDEN_ETIQUETAS_ZS) - len(set(HIPOTESIS_FASE4[formulacion].values())),
        "segundos": registro["segundos"], "segundos_por_texto": registro["segundos_por_texto"],
        **{f"auc {e}": v for e, v in aucs.items()},
    }
```

## 5. Reconocimiento de entidades y referentes

Se compararon tres modelos NER y dos estrategias de agregación de subpalabras. La configuración seleccionada fue **BETO-NER con `first`**. La auditoría contrasta cobertura de referentes con ruido y revisa variaciones de mayúsculas y nombres de clubes. La cobertura del diccionario apoya el control de calidad y su universo depende de los nombres y alias disponibles.

**Ubicación de menciones y recuperación de intervalos en el texto**

```python
def recortar_bordes(texto, inicio, fin):
    while inicio < fin and texto[inicio] in SIGNOS_BORDE:
        inicio += 1
    while fin > inicio and texto[fin - 1] in SIGNOS_BORDE:
        fin -= 1
    return inicio, fin


def inferir_ner(clave, estrategia):
    espec = MODELOS_FASE4[clave]
    configuracion = {
        "nombre": f"ner_{estrategia}", "modelo": espec["id"], "revision": espec["revision"],
        "estrategia_agregacion": estrategia, "max_longitud": espec["max_longitud"],
        "entrada": HUELLA_ENTRADA_FASE4, "transformers": transformers.__version__,
        "codigo": VERSION_CODIGO_INFERENCIA,
    }
    ruta = ruta_cache(clave, configuracion, "csv.gz")
    ruta_meta = CACHE_INFERENCIA_FASE4 / (ruta.name.removesuffix(".csv.gz") + ".json")
    if ruta.exists() and ruta_meta.exists():
        meta = json.loads(ruta_meta.read_text(encoding="utf-8"))
        if meta.get("configuracion") == configuracion:
            registrar_ejecucion(clave, configuracion, meta, True)
            return pd.read_csv(ruta, keep_default_na=False), meta

    inicio_reloj = time.perf_counter()
    tokenizador, modelo = cargar_modelo(clave, AutoModelForTokenClassification)
    longitud = longitud_admitida(modelo, clave)
    limite = longitud - tokenizador.num_special_tokens_to_add(pair=False)
    fragmentos = fragmentar_corpus(tokenizador, limite)
    extractor = pipeline(
        "token-classification", model=modelo, tokenizer=tokenizador,
        aggregation_strategy=estrategia, device=DISPOSITIVO_PIPELINE_FASE4,
    )
    textos = TEXTOS_FASE4.tolist()
    salidas = extractor([textos[p][a:b] for p, a, b, _ in fragmentos], batch_size=LOTE_NER)
    filas = []
    for (posicion, a, _, _), entidades in zip(fragmentos, salidas):
        texto = textos[posicion]
        for entidad in entidades:
            inicio, fin = recortar_bordes(texto, a + int(entidad["start"]), a + int(entidad["end"]))
            if inicio >= fin:
                continue
            filas.append(
                {
                    "cantico_id": IDS_FASE4[posicion], "inicio": inicio, "fin": fin,
                    "mencion": texto[inicio:fin], "tipo": entidad["entity_group"],
                    "puntuacion": float(entidad["score"]),
                }
            )
    entidades = pd.DataFrame(
        filas, columns=["cantico_id", "inicio", "fin", "mencion", "tipo", "puntuacion"]
    )
    parametros = sum(p.numel() for p in modelo.parameters())
    meta = meta_fragmentos(fragmentos, longitud, parametros, time.perf_counter() - inicio_reloj)
    entidades.to_csv(ruta, index=False, encoding="utf-8")
    ruta_meta.write_text(
        json.dumps({**meta, "configuracion": configuracion}, ensure_ascii=False, indent=1),
        encoding="utf-8",
    )
    registrar_ejecucion(clave, configuracion, meta, False)
    del extractor, modelo, tokenizador
    liberar_memoria()
    return entidades, meta
```

**Controles de ruido y cobertura de referentes**

```python
def solapa(inicio_a, fin_a, inicio_b, fin_b):
    return inicio_a < fin_b and inicio_b < fin_a


def rasgos_mencion(tabla):
    tabla = tabla.copy()
    normalizada = tabla["mencion"].map(normalizar_alineado)
    tabla["corta_palabra"] = [
        (i > 0 and texto_por_id[c][i - 1].isalnum()) or (f < len(texto_por_id[c]) and texto_por_id[c][f].isalnum())
        for c, i, f in tabla[["cantico_id", "inicio", "fin"]].itertuples(index=False)
    ]
    tabla["cruza_linea"] = tabla["mencion"].str.contains("\n")
    tabla["ruido_evidente"] = (
        normalizada.isin({normalizar_alineado(f) for f in FORMULAS_NO_ENTIDAD})
        | normalizada.isin(STOPWORDS_NLTK_ES)
        | normalizada.str.len().le(2)
    )
    return tabla


def cobertura_referentes(tabla, referencia):
    """Proporción de menciones del diccionario solapadas por alguna entidad del modelo."""
    por_texto = {c: g[["inicio", "fin", "tipo"]].to_numpy() for c, g in tabla.groupby("cantico_id")}
    cubiertas, tipos = [], []
    for c, i, f in referencia[["cantico_id", "inicio", "fin"]].itertuples(index=False):
        tipo = next(
            (t for a, b, t in por_texto.get(c, []) if solapa(i, f, int(a), int(b))), None
        )
        cubiertas.append(tipo is not None)
        tipos.append(tipo)
    return np.array(cubiertas), pd.Series(tipos, dtype="object")
```

## 6. Alcance de los resultados

La cobertura de textos proviene de fuentes digitales seleccionadas. Los resultados sirven para explorar expresiones visibles en el corpus, plantear hipótesis y orientar la anotación humana. Las salidas de los modelos describen predicciones sobre los textos, sin inferir emociones personales de las hinchadas.